---
# Section 1: Pre-processing
---

## ***Quy trình thực hiện***
### Đọc dữ liệu và khám phá sơ bộ

- `categories.csv`
- `products.csv`
- `reviews.csv`
- `stores.csv`

### I. Làm sạch dữ liệu

Thực hiện lần lượt cho từng bảng:
1. `categories.csv`
2. `products.csv`
3. `reviews.csv`
4. `stores.csv`

#### Với mỗi bảng:
- **Thống kê cơ bản**: kích thước , thông tin tổng quan (info)
- **Kiểm tra trùng lặp** 
- **Kiểm tra giá trị thiếu**
- **Lọc dữ liệu**: thực hiện các thao tác làm sạch cần thiết cho từng thuộc tính.

### II. Chuẩn hóa dữ liệu

1. **Kiểm tra mapping giữa các bảng**  
   - Xác định các trường có thể join với nhau  
   - Tìm các bản ghi **không mapping được**  
   → *Phục vụ việc thiết kế database và xử lý sau này*

2. **Kiểm tra sản phẩm thuộc nhiều category**  
   - Xác định xem có sản phẩm nào thuộc **nhiều hơn 1 category** hay không

3. **Kiểm tra tính đúng đắn của category**  
   - Đánh giá xem sản phẩm đã được phân loại đúng chưa  
   - Áp dụng các phương pháp xử lý nếu sai, ví dụ:  
     - TF-IDF  
     - (các phương pháp NLP / similarity khác)
     


---
## Đọc dữ liệu và khám phá sơ bộ

Các bước thực hiện: 
- Đọc 4 bảng dữ liệu: categories, products, reviews, stores.
- Thực hiện thống kê cơ bản, kiểm tra trùng, thiếu và giá trị unique cho từng bảng.
- Kiểm tra khả năng ánh xạ giữa các bảng.
- Kiểm tra lại phân loại category của sản phẩm ở mức đơn giản.

### Đọc dữ liệu

In [184]:
import pandas as pd
from pathlib import Path
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [185]:
data_dir = Path("../data")

categories_path = data_dir / "categories.csv"
products_path = data_dir / "products.csv"
reviews_path = data_dir / "reviews.csv"
stores_path = data_dir / "stores.csv"

print("Đọc dữ liệu từ:")
print(f"- categories: {categories_path}")
print(f"- products:   {products_path}")
print(f"- reviews:    {reviews_path}")
print(f"- stores:     {stores_path}")

df_category = pd.read_csv(categories_path)
df_product = pd.read_csv(products_path)
df_review = pd.read_csv(reviews_path)
df_store = pd.read_csv(stores_path)

Đọc dữ liệu từ:
- categories: ../data/categories.csv
- products:   ../data/products.csv
- reviews:    ../data/reviews.csv
- stores:     ../data/stores.csv


In [186]:
df_category.head(5)

,category_id,category_name,parent_category,source_category
0,1882,Dien gia dung,NaN,diengiadung
1,1884,Đồ dùng nhà bếp,Dien gia dung,diengiadung
2,1892,Nồi điện các loại,Đồ dùng nhà bếp,diengiadung
3,1893,Nồi cơm điện,Nồi điện các loại,diengiadung
4,1918,Nồi cơm điện tử,Nồi cơm điện,diengiadung


In [187]:
df_product.head(5)

,product_id,store_id,category_id,product_name,product_url,brand,description,price,original_price,discount_percent,sold_count,rating_avg,review_count,source_category
0,273634419,1.0,1918,Nồi cơm điện tử Elmich 0.8L RCE-3915 - Hàng Ch...,https://tiki.vn/noi-com-dien-tu-elmich-0-8l-rc...,Elmich,Thông tin sản phẩm ...,1000000,2200000,55,130,5.0,18,diengiadung
1,134550148,1.0,1918,Nồi cơm điện tử Elmich RCE-1789 (1.2 Lít) - Hà...,https://tiki.vn/noi-com-dien-tu-elmich-rce-178...,Elmich,Chất liệu tuyệt đối cho sức khỏeNồi cơm điện t...,1136000,2500000,55,301,4.8,63,diengiadung
2,274106659,1.0,1918,Nồi Cơm Điện Tử Mini Philips HD3170/66 - 600W ...,https://tiki.vn/noi-com-dien-tu-mini-philips-h...,Philips,Nồi cơm điện Dòng 3000 của Philips có dung tíc...,965000,1290000,25,61,4.8,8,diengiadung
3,274965977,1.0,1918,"Nồi cơm điện tử Kangaroo 1.8L model KG18DR12, ...",https://tiki.vn/noi-com-dien-tu-1-8l-model-kg1...,Kangaroo,"Dung tích 1,8LDung tích nồi cơm điện KG18DR12 ...",1210000,2520000,52,16,4.8,5,diengiadung
4,260224713,1.0,1918,Nồi cơm điện cao tần Daewoo DWRC-G411IH 1.8L l...,https://tiki.vn/noi-com-dien-cao-tan-1-8l-daew...,Daewoo,...,1626000,2990000,46,9,3.0,2,diengiadung


In [188]:
df_review.head(5)

,review_id,product_id,user_name,rating,review_text,like_count,review_date,source_category
0,20007421,273634419,Hồ Hữu Nghĩa,5,"Sản phẩm xài rất ok. Nấu nhanh, gọn nhẹ, nhiểu...",0,2024-10-16T03:30:42.000Z,diengiadung
1,20205429,273634419,Le On,5,Cực kì hài lòng,0,2026-01-13T06:29:08.000Z,diengiadung
2,20202180,273634419,Luyen pv,5,Cực kì hài lòng,0,2025-12-30T16:02:30.000Z,diengiadung
3,20191102,273634419,Nguyễn Văn Khang,5,Cực kì hài lòng,0,2025-11-16T08:24:31.000Z,diengiadung
4,20149003,273634419,Hoàng Văn Chương,5,Dùng tốt,0,2025-07-20T07:46:41.000Z,diengiadung


In [189]:
df_store.head(5)

,store_id,store_name,store_rating,follower_count,source_category
0,1,Tiki Trading,4.6754,512930,diengiadung
1,308020,GiaDungNhaThongMinh,4.8750,39,diengiadung
2,29960,CUCKOO Store,4.6289,4894,diengiadung
3,160749,Mishio Kachi Official,4.5030,5396,diengiadung
4,360957,Seka Official Store,4.0000,0,diengiadung


### Biểu đồ tương tác cho toàn bộ cây category

Mục đích: Khám phá rõ hơn mối quan hệ giữa các category, đặc biệt là các category con và category cha, để từ đó có thể thiết kế biểu đồ tương tác phù hợp.

In [190]:
import pandas as pd
import plotly.express as px
import unicodedata
from pathlib import Path
from collections import defaultdict

if "df_category" in globals():
    categories_sunburst = df_category.copy()
else:
    categories_sunburst = pd.read_csv(Path("../data/categories.csv"))

required_cols = {"category_id", "category_name", "parent_category"}
missing_cols = required_cols - set(categories_sunburst.columns)
if missing_cols:
    raise ValueError(f"Thiếu cột bắt buộc: {missing_cols}")

if "source_category" not in categories_sunburst.columns:
    categories_sunburst["source_category"] = ""

sunburst_df = categories_sunburst[["category_id", "category_name", "parent_category", "source_category"]].copy()
sunburst_df["category_id"] = sunburst_df["category_id"].astype(str).str.strip()
sunburst_df["category_name"] = sunburst_df["category_name"].astype(str).str.strip()
sunburst_df["parent_category"] = sunburst_df["parent_category"].fillna("").astype(str).str.strip()
sunburst_df["source_category"] = sunburst_df["source_category"].fillna("").astype(str).str.strip()

# 5 root cần focus
roots_to_focus = [
    "Laptop - May vi tinh - Linh kien",
    "Dien thoai - May tinh bang",
    "Thiet bi so - Phu kien so",
    "Dien gia dung",
    "Dien tu - Dien lanh",
]

def normalize_text(s: str) -> str:
    s = str(s).strip().lower()
    s = s.replace("đ", "d")
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return " ".join(s.split())

# Tạo key để map parent -> child theo ID, tránh lỗi trùng category_name
sunburst_df["name_key"] = sunburst_df["category_name"].apply(normalize_text)
sunburst_df["source_key"] = sunburst_df["source_category"].apply(normalize_text)
sunburst_df["node_id"] = sunburst_df["category_id"]

name_source_to_ids = defaultdict(list)
name_to_ids = defaultdict(list)
for _, row in sunburst_df.iterrows():
    name_source_to_ids[(row["source_key"], row["name_key"])].append(row["node_id"])
    name_to_ids[row["name_key"]].append(row["node_id"])

def resolve_parent_id(row):
    parent_name = row["parent_category"]
    if parent_name == "":
        return ""

    parent_key = normalize_text(parent_name)
    source_key = row["source_key"]

    candidates = list(dict.fromkeys(name_source_to_ids.get((source_key, parent_key), [])))
    if len(candidates) == 1:
        return candidates[0]

    candidates = list(dict.fromkeys(name_to_ids.get(parent_key, [])))
    if len(candidates) == 1:
        return candidates[0]

    return candidates[0] if len(candidates) > 0 else ""

sunburst_df["parent_id"] = sunburst_df.apply(resolve_parent_id, axis=1)

# Build cây parent_id -> child_id
children_map = defaultdict(list)
for _, row in sunburst_df.iterrows():
    parent_id = row["parent_id"]
    if parent_id != "":
        children_map[parent_id].append(row["node_id"])

def get_all_descendants(root_id: str):
    keep = set()
    stack = [root_id]
    while stack:
        cur = stack.pop()
        if cur in keep:
            continue
        keep.add(cur)
        stack.extend(children_map.get(cur, []))
    return keep

for root in roots_to_focus:
    root_key = normalize_text(root)
    root_candidates = list(dict.fromkeys(name_to_ids.get(root_key, [])))

    if len(root_candidates) == 0:
        print(f"Không tìm thấy root trong categories: {root}")
        continue

    top_level_candidates = [
        rid for rid in root_candidates
        if sunburst_df.loc[sunburst_df["node_id"] == rid, "parent_id"].iloc[0] == ""
    ]
    root_id = top_level_candidates[0] if top_level_candidates else root_candidates[0]

    keep_ids = get_all_descendants(root_id)

    df_root = sunburst_df[sunburst_df["node_id"].isin(keep_ids)].copy()
    df_root = df_root[["node_id", "category_name", "parent_id"]].drop_duplicates(subset=["node_id"])

    # đảm bảo root là nút gốc trong chart này
    df_root.loc[df_root["node_id"] == root_id, "parent_id"] = ""

    real_root = df_root.loc[df_root["node_id"] == root_id, "category_name"].iloc[0]

    fig = px.sunburst(
        df_root,
        ids="node_id",
        names="category_name",
        parents="parent_id",
        title=f"Sunburst cho root: {real_root}",
    )
    
    fig.update_layout(
        height=850,
        width=1000,
        margin=dict(t=70, l=10, r=10, b=10),
        uniformtext=dict(minsize=9, mode="hide"),
    )
    fig.show()

## I. Làm sạch dữ liệu

Với từng df `df_category`, `df_product`, `df_review`, `df_store`:
- Thống kê kích thước, kiểu dữ liệu
- Kiểm tra trùng dòng
- Kiểm tra missing
- Lọc dữ liệu.

### `df_category`

#### 1. Thống kê cơ bản

In [191]:
df_category.shape

(497, 4)

In [192]:
df_category.info()

<class 'pandas.DataFrame'>
RangeIndex: 497 entries, 0 to 496
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   category_id      497 non-null    int64
 1   category_name    497 non-null    str  
 2   parent_category  492 non-null    str  
 3   source_category  497 non-null    str  
dtypes: int64(1), str(3)
memory usage: 15.7 KB


#### 2. Check duplicate

In [193]:
df_category[df_category.duplicated(subset=["category_id"], keep=False)]

,category_id,category_name,parent_category,source_category


In [194]:
df_category[df_category.duplicated(subset=["category_name"], keep=False)]

,category_id,category_name,parent_category,source_category
121,5976,Máy sấy quần áo,Thiết bị gia đình,diengiadung
166,3863,Máy sấy quần áo,Dien tu - Dien lanh,dientu_dienlanh


Nhóm sẽ kiểm tra dữ liệu ở 2 mục này sau để quyết định giữ category ở bên nào

In [195]:
# in 5 sản phẩm của category "Máy sấy quần áo" ở category_id = 5976
df_product[df_product["category_id"] == 5976].head(5)

,product_id,store_id,category_id,product_name,product_url,brand,description,price,original_price,discount_percent,sold_count,rating_avg,review_count,source_category
11098,278872428,1.0,5976,Tủ sấy quần áo Elmich CDE-8641 - Hàng Chính Hãng,https://tiki.vn/tu-say-quan-ao-elmich-cde-8641...,Elmich,Kích thước rộng rãi: Tủ sấy có kích thước tổng...,1358000,2988000,55,27,5.0,2,diengiadung
11099,278893185,1.0,5976,Tủ sấy quần áo Elmich CDE-1296 - Hàng Chính Hãng,https://tiki.vn/tu-say-quan-ao-elmich-cde-1296...,Elmich,Câu Chuyện Sản Phẩm Tủ sấy quần áo Elmich CDE1...,1540000,3388000,55,3,0.0,0,diengiadung
11100,279165505,292665.0,5976,"Tủ Sấy Quần Áo Elmich CDE1296, Hàng Chính Hãng...",https://tiki.vn/tu-say-quan-ao-elmich-cde1296-...,Elmich,"Tủ Sấy Quần Áo Elmich CDE1296, Hàng Chính Hãng...",2550000,2550000,0,0,0.0,0,diengiadung
11101,278989424,1.0,5976,"Máy sấy khuẩn quần áo LocknLocK,khử khuẩn tia ...",https://tiki.vn/may-say-khuan-quan-ao-locknloc...,LocknLock,Máy sấy khuẩn quần áo LocknLock Cloth steriliz...,1389500,1985000,30,0,0.0,0,diengiadung
11102,278877812,297289.0,5976,Tủ sấy quần áo KORENO KN - 569 LOẠI TO công su...,https://tiki.vn/tu-say-quan-ao-koreno-kn-569-l...,KORENO,CÁC ĐẶC ĐIỂM NỔI BẬT:- Khối lượng sấy quần áo ...,1650000,1650000,0,0,0.0,0,diengiadung


In [196]:
# in 5 sản phẩm của category "Máy sấy quần áo" ở category_id = 3863
df_product[df_product["category_id"] == 3863].head(5)

,product_id,store_id,category_id,product_name,product_url,brand,description,price,original_price,discount_percent,sold_count,rating_avg,review_count,source_category


Ta nhận thấy "Máy sấy quần áo" ở `category_id` = 3863 thực chất là một category rỗng

-> Tiến hành loại bỏ

In [197]:
# xóa category_id = 3863
df_category = df_category[df_category["category_id"] != 3863].copy()

In [198]:
#kiểm tra lại
df_category[df_category.duplicated(subset=["category_name"], keep=False)]

,category_id,category_name,parent_category,source_category


#### 3. Check missing

In [199]:
# kiểm tra missing toàn bộ thuộc tính
print("Missing values in df_category:")
print(df_category.isnull().sum())

Missing values in df_category:
category_id        0
category_name      0
parent_category    5
source_category    0
dtype: int64


Đây là 5 root category (5 nhánh thư mục gốc, không cần loại xử lý)

#### 4. Lọc dữ liệu

`category_id`    
- Đang có kiểu dữ liệu int64, đã phù hợp để dùng làm key.

`parent_category`
- Loại bỏ khoảng trống.

In [200]:
df_category['parent_category'] = df_category['parent_category'].fillna('').astype(str).str.strip()

`category_name`
- Loại bỏ khoảng trống.
- Do root category còn là tên gốc của folder crawl nên chưa đồng bộ. Em sẽ chuyển về dạng chuẩn như các category khác.


In [201]:
df_category['category_name'] = df_category['category_name'].astype(str).str.strip()

Kiểm tra root category

In [202]:
root_categories = df_category[df_category["parent_category"] == ""]

print("Root categories:")
display(root_categories)

Root categories:


,category_id,category_name,parent_category,source_category
0,1882,Dien gia dung,,diengiadung
138,1789,Dien thoai - May tinh bang,,dienthoai_maytinhbang
144,4221,Dien tu - Dien lanh,,dientu_dienlanh
198,1846,Laptop - May vi tinh - Linh kien,,laptop_mayvitinh_linhkien
291,1815,Thiet bi so - Phu kien so,,thietbiso_phukienso


In [203]:
root_category_mapping = {
    "Dien gia dung": "Điện gia dụng",
    "Dien thoai - May tinh bang": "Điện thoại - Máy tính bảng",
    "Dien tu - Dien lanh": "Điện tử - Điện lạnh",
    "Laptop - May vi tinh - Linh kien": "Laptop - Máy vi tính - Linh kiện",
    "Thiet bi so - Phu kien so": "Thiết bị số - Phụ kiện số",
}

df_category['category_name'] = df_category['category_name'].replace(root_category_mapping)

Kiểm tra lại

In [204]:
root_categories = df_category[df_category["parent_category"] == ""]

print("Root categories:")
display(root_categories)

Root categories:


,category_id,category_name,parent_category,source_category
0,1882,Điện gia dụng,,diengiadung
138,1789,Điện thoại - Máy tính bảng,,dienthoai_maytinhbang
144,4221,Điện tử - Điện lạnh,,dientu_dienlanh
198,1846,Laptop - Máy vi tính - Linh kiện,,laptop_mayvitinh_linhkien
291,1815,Thiết bị số - Phụ kiện số,,thietbiso_phukienso


### `df_product`

#### 1. Thống kê cơ bản

In [205]:
df_product.shape

(55883, 14)

In [206]:
df_product.info()

<class 'pandas.DataFrame'>
RangeIndex: 55883 entries, 0 to 55882
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   product_id        55883 non-null  int64  
 1   store_id          55023 non-null  float64
 2   category_id       55883 non-null  int64  
 3   product_name      55883 non-null  str    
 4   product_url       55883 non-null  str    
 5   brand             55881 non-null  str    
 6   description       55830 non-null  str    
 7   price             55883 non-null  int64  
 8   original_price    55883 non-null  int64  
 9   discount_percent  55883 non-null  int64  
 10  sold_count        55883 non-null  int64  
 11  rating_avg        55883 non-null  float64
 12  review_count      55883 non-null  int64  
 13  source_category   55883 non-null  str    
dtypes: float64(2), int64(7), str(5)
memory usage: 6.0 MB


#### 2. Check duplicate

In [207]:
# kiểm tra product_url và brand và price có bị trùng lặp không
df_product[df_product.duplicated(subset=["product_url"], keep=False)]

,product_id,store_id,category_id,product_name,product_url,brand,description,price,original_price,discount_percent,sold_count,rating_avg,review_count,source_category


Không có trùng lặp

#### 3. Check missing

In [208]:
# kiểm tra missing
print("Missing values in df_product:")
print(df_product.isnull().sum())

Missing values in df_product:
product_id            0
store_id            860
category_id           0
product_name          0
product_url           0
brand                 2
description          53
price                 0
original_price        0
discount_percent      0
sold_count            0
rating_avg            0
review_count          0
source_category       0
dtype: int64


Các dữ liệu missing:
- `store_id`: không thu thập được dữ liệu store của các sản phẩm.
- `brand`: sản phẩm NoBrand
- `description`: không có mô tả, vấn đề này không quá ảnh hưởng đến quá trình phân tích

#### 4. Lọc dữ liệu

`product_id`
- Đang có kiểu dữ liệu int64, đã phù hợp để dùng làm key.

`store_id`
- Đang là float, chuyển về int.

In [209]:
df_product["store_id"] = df_product["store_id"].astype("Int64") 

`product_name`
- Loại bỏ khoảng trống.
- Lowercase.

In [210]:
df_product["product_name"] = df_product["product_name"].astype(str).str.strip()

df_product["product_name"] = df_product["product_name"].str.lower()

`brand`
- Loại bỏ khoảng trống.
- Lowercase.
- Khám phá xem có vấn đề gì khác không.

In [211]:
df_product["brand"] = df_product["brand"].astype(str).str.strip().str.lower()

df_product["brand"] = df_product["brand"].str.lower()

Khám phá thêm

In [212]:
#brand unique values
print("Unique brands:")
for brand in df_product["brand"].unique():
    print(brand)

Unique brands:
elmich
philips
kangaroo
daewoo
olivo
bear
sunhouse
cuckoo
toshiba
mishio
seka
locknlock
lock&lock
tiger
sharp
ht sys
tefal
haatz
happy home pro
kuchenzimmer
panasonic
magic eco
nagakawa
ladomax
cuchen
nk media
midea
gali
xiaomi
saiko
tiross
masuto
bluestone
hitachi
korea king
kitacoom
electrolux
zojirushi
bennix
dreamer
beko
sfiapr
benny
hyundai
joseph joseph
smartcook
casper
truehome
sato
hlt@chl
guckdo
anh lam store
hamm
srapp
jlpl@l
eaststar
kalpen
bigsun
fujika
kipor
hasuka
comet
osako
geidea
galuz
pengo
aidi
sowun
fumak
iviaivia cook
vonshef
unie
đạt tường
russell hobbs
orkin
lebenlang
arber
kitchen flower
kim cương
gugkdd
tiger queen
kiwa
alaska
vinsun
joyoung
highgate
rainy
instant pot
miniin
seoul
eurosun
quick eco
gume
crockpot
kuchen
klarstein
fissler
faster
khaluck
lumena
supor
x linh chi store
life360 amazing experience
green cook
kuvings
av crystan
apelek
khoncc
lorente
matika
d danido
dongyuan
morphy richards
unold
iris ohyama
tahawa
vera
weishi
creen
bbcoo

Quan sát:
- Nhận thấy có nhiều giá trị lạ.
- Không có nhiễu nào rõ ràng cần phải loại bỏ.

Tìm hiểu:
- Em đã đi tìm hiểu các shop này và nhận thấy: có đa dạng các thương hiệu từ global đến local.
- Nhiều thương hiệu đồ nội địa Trung có tên thương hiệu mang tính ngẫu nhiên, không quy luật nên giá trị lạ đã phát hiện không phải nhiễu.
- Một số cửa hàng không điền đúng nhãn hiệu sản phẩm vào "brand" mà điền tên shop mình

`description`
- Loại bỏ khoảng trống.
- Lowercase.

In [213]:
df_product["description"] = df_product["description"].astype(str).str.strip().str.lower()

df_product["description"] = df_product["description"].str.lower()

`source_category`

- Thuộc tính từ script crawl, tạm thời giữ nguyên cột này 

`price`, `original_price`, `discount_percent`

- Kiểm tra logic (`discount_percent` có đúng không, `original_price` và `price` có bất thường không)


In [214]:
# 1.kiểm tra giá
if (df_product["original_price"] < 0).any():
    print("Có sản phẩm có giá gốc âm")

if (df_product["original_price"] == 0).any():
    print("Có sản phẩm có giá gốc bằng 0")

if (df_product["price"] < 0).any():
    print("Có sản phẩm có giá âm")

if (df_product["price"] > df_product["original_price"]).any():
    print("Có sản phẩm có giá sale lớn hơn giá gốc")


# 2. kiểm tra discount
if (df_product["discount_percent"] < 0).any():
    print("Có sản phẩm có phần trăm giảm giá âm")

if (df_product["discount_percent"] > 100).any():
    print("Có sản phẩm có phần trăm giảm giá > 100%")


# 3. kiểm tra discount_percent có khớp với original_price 
# và price không
valid = df_product["original_price"] > 0

calc_discount = (
    (df_product.loc[valid, "original_price"] - df_product.loc[valid, "price"])
    / df_product.loc[valid, "original_price"]
    * 100
)
mismatch = (calc_discount.round(2) != df_product.loc[valid, "discount_percent"])

if mismatch.any():
    print("Có sản phẩm có discount không khớp")

Có sản phẩm có giá sale lớn hơn giá gốc
Có sản phẩm có discount không khớp


Phát hiện bất thường, tiến hành phân tích

In [215]:
#in ra sản phẩm có giá sale lớn hơn giá gốc
invalid_price = df_product[df_product["price"] > df_product["original_price"]]
if not invalid_price.empty:
    print("Sản phẩm có giá sale lớn hơn giá gốc:")
    print(f"Số lượng: {len(invalid_price)}")
    display(invalid_price[["product_id", "product_name", "original_price", "price"]])

# in ra sản phâm có discount không khớp
valid = df_product["original_price"] > 0
calc_discount = (
    (df_product.loc[valid, "original_price"] - df_product.loc[valid, "price"])
    / df_product.loc[valid, "original_price"]
    * 100
)
mismatch = (calc_discount.round(2) != df_product.loc[valid, "discount_percent"])
if mismatch.any():
    print("Sản phẩm có discount không khớp:")
    print(f"Số lượng: {mismatch.sum()}")
    display(df_product.loc[valid].loc[mismatch, ["product_id", "product_name", "original_price", "price", "discount_percent"]])

Sản phẩm có giá sale lớn hơn giá gốc:
Số lượng: 29


,product_id,product_name,original_price,price
2865,47885195,bếp gas đôi rinnai rv377g xám hàng chính hãng,900000,925000
2957,498200,bếp ga dương đôi rinnai rv-6double glass(sp) -...,1900000,2060000
2997,496817,bếp gas dương đôi rinnai rv-367(g)n – đen- hãn...,850000,925000
3077,498202,bếp ga dương đôi rinnai rv-6double glass(b) - ...,1900000,2060000
6151,279003324,"máy pha cà phê bán tự động espresso, cappuccin...",4120000,4223000
6386,249746506,máy xay sửa hạt olivo x20 plus thương hiệu mỹ ...,2499000,2799000
6463,122576690,máy làm sữa chua hương vị truyền thống song an...,380000,500000
6638,276153010,hàng nhập khẩu. máy sấy thực phẩm gia đình 5 k...,1914000,2183000
6639,276152775,máy sấy thực phẩm gia đình 5 khay thương hiệu ...,1914000,2169000
6824,242453901,máy nấu chậm sous vide biolomix sv-8006 công s...,2475000,2857000


Sản phẩm có discount không khớp:
Số lượng: 11736


,product_id,product_name,original_price,price,discount_percent
0,273634419,nồi cơm điện tử elmich 0.8l rce-3915 - hàng ch...,2200000,1000000,55
1,134550148,nồi cơm điện tử elmich rce-1789 (1.2 lít) - hà...,2500000,1136000,55
2,274106659,nồi cơm điện tử mini philips hd3170/66 - 600w ...,1290000,965000,25
3,274965977,"nồi cơm điện tử kangaroo 1.8l model kg18dr12, ...",2520000,1210000,52
4,260224713,nồi cơm điện cao tần daewoo dwrc-g411ih 1.8l l...,2990000,1626000,46
...,...,...,...,...,...
55807,6815007,bộ chia cổng hdmi 1 cổng ra 2 cổng hỗ trợ full...,548000,529000,3
55808,2002923,bộ chia hdmi ugreen ra 4 cổng hdmi 40202 - hàn...,1080000,750000,31
55810,1998419,cáp máy in ugreen 10329 (5m) - hàng chính hãng,250000,99000,60
55811,1998349,cáp máy in ugreen 10352 (5m) - hàng chính hãng...,139000,129000,7


Về các sản phẩm có `price` > `original_price`

Lý do:
- Vấn đề trong lúc crawl dữ liệu.

Giải pháp: 
- Do `price` chuẩn hơn trong trường hợp này nên em sẽ assign `original_price` = `price` để các phần phân tích không bị nhầm insight

In [216]:
#gán original_price = price nếu original_price < price
invalid_price = df_product[df_product["price"] > df_product["original_price"]]
if not invalid_price.empty:
    df_product.loc[invalid_price.index, "original_price"] = df_product.loc[invalid_price.index, "price"]
    print(f"Đã gán original_price = price cho {len(invalid_price)} sản phẩm có giá sale lớn hơn giá gốc.")

Đã gán original_price = price cho 29 sản phẩm có giá sale lớn hơn giá gốc.


Về các sản phẩm có ``discount_percent` không khớp  
 
Lý do:
- Do bên cạnh discount gốc còn thêm nhiều discount khác: đến từ shop, sàn,...
- `discount_percent` chỉ là % hiển thị mà script scrawl được
 
Giải pháp: 
- Nếu `price` = `original_price` * (100 - `discount_percent`) += 5% thì chấp nhận.
- Nếu khoảng cách lớn hơn ngưỡng 5% thì tính lại `discount_percent`

Tại sao không tính lại toàn bộ:
- Vì đây là dữ liệu trực quan nhất mà người dùng nhìn thấy được từ Tiki, con số trực quan cũng ảnh hưởng một phần đến quyết định (không phải người mua nào cũng sẽ kiểm tra lại % này)
- Nên trừ lý do bất khả kháng (chênh lệch nhiều), còn lại em xin phép được giữ nguyên.

In [217]:
# nếu discount_percent sai số += 5% thì giữ nguyên, nếu sai số > 5% thì gán discount_percent = discount tính được từ price và original_price
valid = df_product["original_price"] > 0
calc_discount = (
    (df_product.loc[valid, "original_price"] - df_product.loc[valid, "price"])
    / df_product.loc[valid, "original_price"]
    * 100
)
mismatch = (calc_discount.round(2) != df_product.loc[valid, "discount_percent"])
if mismatch.any():
    mismatch_percent = (calc_discount - df_product.loc[valid, "discount_percent"]).abs()
    to_fix = mismatch_percent > 5
    if to_fix.any():
        df_product.loc[valid].loc[to_fix, "discount_percent"] = calc_discount.loc[to_fix].round(2)
        print(f"Đã gán lại discount_percent cho {to_fix.sum()} sản phẩm có sai số > 5% so với giá gốc và giá sale.")

`sold_count`, `rating_avg` và `review_count` đã phù hợp. Không cần tiền xử lý thêm

### `df_review`

#### 1. Thống kê cơ bản

In [218]:
df_review.shape

(158126, 8)

In [219]:
df_review.info()

<class 'pandas.DataFrame'>
RangeIndex: 158126 entries, 0 to 158125
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype
---  ------           --------------   -----
 0   review_id        158126 non-null  int64
 1   product_id       158126 non-null  int64
 2   user_name        158126 non-null  str  
 3   rating           158126 non-null  int64
 4   review_text      158126 non-null  str  
 5   like_count       158126 non-null  int64
 6   review_date      158126 non-null  str  
 7   source_category  158126 non-null  str  
dtypes: int64(4), str(4)
memory usage: 9.7 MB


#### 2. Check duplicate

In [220]:
df_review[df_review.duplicated(subset=["review_id"], keep=False)]

,review_id,product_id,user_name,rating,review_text,like_count,review_date,source_category


In [221]:
#xóa các dòng trùng lặp
df_review = df_review.drop_duplicates(subset=["review_id"], keep="first").reset_index(drop=True)

#kiểm tra lại
if df_review.duplicated(subset=["review_id"], keep=False).any():
    print("There are duplicate rows in the 'review_id' column.")
else:
    print("No duplicate rows found in the 'review_id' column.")

No duplicate rows found in the 'review_id' column.


#### 3. Check missing

In [222]:
# kiểm tra missing
print("Missing values in 'df_review':")
print(df_review["review_id"].isnull().sum())

Missing values in 'df_review':
0


#### 4. Lọc dữ liệu

`review_id`
- Đang có kiểu dữ liệu int64, đã phù hợp để dùng làm key.

`product_id`    
- Đang có kiểu dữ liệu int64, đã phù hợp để dùng làm key.

`user_name`
- Loại bỏ khoảng trống.
- Hiện đang có kiểu string, dữ liệu này mang tính cá nhân và không có nhiều giá trị phân tích, em sẽ không tiền xử lý sâu

In [223]:
df_review["user_name"] = df_review["user_name"].astype(str).str.strip().str.lower()

`review_text`
- Kiểm tra rỗng.
- Loại bỏ khoảng trống.
- Loại bỏ emoji, ký tự rác.
- Chuẩn hóa unicode.

In [224]:
df_review["review_text"] = df_review["review_text"].fillna("").astype(str).str.strip().str.lower()

(df_review["review_text"].str.len() == 0).sum()

np.int64(0)

In [225]:
# loại bỏ emoji và kí tự đặc biệt trong review_text
import re

def remove_emoji_and_special_characters(text):
    # Loại bỏ emoji
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags (iOS)
        "]+",
        flags=re.UNICODE,
    )
    text = emoji_pattern.sub(r"", text)

    # Loại bỏ ký tự đặc biệt (chỉ giữ lại chữ và số)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)

    return text

df_review["cleaned_review_text"] = df_review["review_text"].apply(remove_emoji_and_special_characters)

`review_date`
- Chuyển về datetime

In [226]:
df_review["review_date"] = pd.to_datetime(df_review["review_date"], errors="coerce")

`likes_count` và `source_category` không cần xử lý thêm

### `df_store`

#### 1. Thống kê cơ bản

In [227]:
df_store.info()

<class 'pandas.DataFrame'>
RangeIndex: 1446 entries, 0 to 1445
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   store_id         1446 non-null   int64  
 1   store_name       1446 non-null   str    
 2   store_rating     1446 non-null   float64
 3   follower_count   1446 non-null   int64  
 4   source_category  1446 non-null   str    
dtypes: float64(1), int64(2), str(2)
memory usage: 56.6 KB


#### 2. Check duplicate

In [228]:
df_category[df_category.duplicated(subset=["category_name"], keep=False)]

,category_id,category_name,parent_category,source_category


In [229]:
#xóa các dòng trùng lặp
df_category = df_category.drop_duplicates(subset=["category_name"], keep="first").reset_index(drop=True)

#kiểm tra lại
if df_category.duplicated(subset=["category_name"], keep=False).any():
    print("There are duplicate rows in the 'category_name' column.")
else:
    print("No duplicate rows found in the 'category_name' column.")

No duplicate rows found in the 'category_name' column.


#### 3. Check missing

In [230]:
# kiểm tra missing
print("Missing values in 'df_review':")
print(df_review.isnull().sum())

Missing values in 'df_review':
review_id              0
product_id             0
user_name              0
rating                 0
review_text            0
like_count             0
review_date            0
source_category        0
cleaned_review_text    0
dtype: int64


#### 4. Lọc dữ liệu

`store_id`
- Đang có kiểu dữ liệu int64, đã phù hợp để dùng làm key.

`store_name`
- Loại bỏ khoảng trống.
- Lowercase

In [231]:
df_store["store_name"] = df_store["store_name"].astype(str).str.strip().str.lower()

df_store["store_name"] = df_store["store_name"].str.lower()

`store_rating`

- Kiểm tra giá trị

In [232]:
df_store["store_rating"].describe()

count    1446.000000
mean        3.222462
std         2.018953
min         0.000000
25%         0.000000
50%         4.340500
75%         4.644325
max         5.000000
Name: store_rating, dtype: float64

`follower_count` và `source_category` không cần tiền xử lý thêm

## II. Chuẩn hóa dữ liệu

### 1. Kiểm tra mapping giữa các bảng

- `df_product.product_id` so với `df_review.product_id`
- `df_product.store_id` so với `df_store.store_id`
- `df_product.category_id` so với `df_category.category_id`

Mục tiêu: tìm ra các bản ghi không mapping được để sau này lưu ý khi tạo database.

In [233]:
def print_missing(left_df, right_df, key, left_name, right_name):
    left_ids = set(left_df[key].dropna().unique())
    right_ids = set(right_df[key].dropna().unique())
    
    missing = left_ids - right_ids
    
    if len(missing) > 0:
        print(f"\n=== {key}: có ở {left_name} nhưng KHÔNG có ở {right_name} ===")
        print(f"Số lượng: {len(missing)}")
        print("Ví dụ (tối đa 10):", list(missing)[:10])
        
        # show full row cho dễ debug
        display(left_df[left_df[key].isin(list(missing)[:10])])
    else:
        print(f"\n=== {key}: OK ({left_name} ⊆ {right_name}) ===")


# 1. reviews → products
print_missing(df_review, df_product, "product_id", "reviews", "products")

# 2. products → stores
print_missing(df_product, df_store, "store_id", "products", "stores")

# 3. products → categories
print_missing(df_product, df_category, "category_id", "products", "categories")


=== product_id: OK (reviews ⊆ products) ===

=== store_id: OK (products ⊆ stores) ===

=== category_id: OK (products ⊆ categories) ===


### 2. Kiểm tra product thuộc nhiều hơn 1 category

Mục tiêu: tìm những product (theo `product_id`) đang gắn với từ 2 `category_id` trở lên → có khả năng lỗi gán category hoặc trùng dòng không nhất quán.

In [234]:
# === Product thuộc nhiều category ===

tmp = df_product[["product_id", "category_id"]].drop_duplicates()

multi_cat = (
    tmp.groupby("product_id")["category_id"]
    .nunique()
    .reset_index(name="category_count")
    .query("category_count > 1")
)

print("Số product thuộc nhiều category:", len(multi_cat))

if not multi_cat.empty:
    detail = multi_cat.merge(tmp, on="product_id")
    
    if "product_name" in df_product.columns:
        detail = detail.merge(
            df_product[["product_id", "product_name"]].drop_duplicates(),
            on="product_id",
            how="left"
        )
    
    display(detail.sort_values("category_count", ascending=False).head(50))

Số product thuộc nhiều category: 0


### 3. Kiểm tra lại category của sản phẩm (dựa trên nội dung text)

Ý tưởng đơn giản:
- Dùng tên sản phẩm (hoặc mô tả) để tách từ khóa
- Gom nhóm theo `category_id` (và/hoặc tên category)
- Đếm tần suất từ xuất hiện trong từng category → xem top từ khóa đặc trưng
- Dựa vào trực quan, phát hiện sản phẩm có tên không phù hợp với category.

(Đây là phiên bản đơn giản hơn tf-idf, dễ chạy và hiểu.)

Kiểm tra một số category

In [235]:
df_category[["category_id", "category_name"]].head(20)

,category_id,category_name
0,1882,Điện gia dụng
1,1884,Đồ dùng nhà bếp
2,1892,Nồi điện các loại
3,1893,Nồi cơm điện
4,1918,Nồi cơm điện tử
5,1919,Nồi cơm nắp gài
6,1920,Nồi cơm nắp rời
7,2271,Nồi cơm dung tích lớn
8,1894,Nồi áp suất điện
9,1895,Nồi lẩu điện


 Check lại category bằng keyword từ product_name 

In [236]:
# 1. Chuẩn bị text (đơn giản)
df = df_product[["product_id", "category_id", "product_name"]].dropna()

# 2. Vectorize (đếm từ)
vectorizer = CountVectorizer(max_features=1000, stop_words="english")
X = vectorizer.fit_transform(df["product_name"])

# 3. Tạo DataFrame từ khóa
keywords_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
keywords_df["category_id"] = df["category_id"].values

# 4. Lấy top keyword mỗi category
top_keywords = (
    keywords_df.groupby("category_id")
    .sum()
    .apply(lambda x: x.sort_values(ascending=False).head(10).index.tolist(), axis=1)
)

# 5. Hiển thị
print("Top keywords mỗi category:")
display(top_keywords.head(10))

Top keywords mỗi category:


category_id
1794    [bảng, máy, tính, hàng, chính, hãng, wifi, xia...
1795    [hãng, hàng, chính, điện, thoại, 128gb, xiaomi...
1796    [4g, điện, hãng, chính, hàng, thoại, màu, qc, ...
1803    [hàng, nghe, máy, nhạc, chính, hãng, bluetooth...
1813    [loa, hàng, chính, hãng, tính, vi, soundmax, m...
1819    [hàng, hãng, chính, cho, điện, thoại, iphone, ...
1820    [thẻ, nhớ, hàng, chính, hãng, micro, 10, sd, 1...
1828    [usb, hàng, hãng, chính, 128gb, ultra, type, g...
1831    [chuột, lót, hàng, hãng, chính, miếng, bàn, pa...
1832    [laptop, nhiệt, tản, hàng, đế, hãng, chính, ch...
dtype: object

Chuẩn bị phụ kiện

In [237]:
accessory_phone = [
    # bảo vệ
    "ốp", "ốp lưng", "case", "bao da", "flip cover", "cường lực", "kính", "miếng dán", "dán màn hình",
    
    # sạc
    "cáp", "dây sạc", "củ sạc", "adapter", "sạc nhanh", "sạc không dây", "dock sạc",
    
    # âm thanh
    "tai nghe", "earphone", "headphone", "airpods", "buds",
    
    # khác
    "giá đỡ", "kẹp điện thoại", "gậy selfie", "ring", "móc", "dây đeo",
    "pin dự phòng", "power bank", "sạc dự phòng",
    "sim", "khay sim", "que chọc sim"
]

accessory_laptop = [
    "chuột", "mouse", "bàn phím", "keyboard", "pad chuột",
    "dock", "hub", "usb hub", "type c hub",
    "cáp", "hdmi", "displayport", "vga",
    "adapter", "củ sạc", "sạc laptop",
    "quạt tản nhiệt", "đế tản nhiệt",
    "ram", "ssd", "hdd", "ổ cứng",
    "webcam", "micro", "loa",
    "balo", "túi chống sốc", "túi laptop"
]

accessory_appliance = [
    "remote", "điều khiển", "điều khiển từ xa",
    "board", "mạch", "bo mạch",
    "cảm biến", "sensor",
    "dây điện", "jack", "đầu nối",
    "ống", "ống dẫn", "ống nước",
    "van", "lọc", "lưới lọc",
    "block", "motor", "quạt",
    "bánh răng", "trục",
    "phụ kiện", "linh kiện", "spare", "replacement"
]

accessory_home = [
    "nắp", "nắp nồi", "vung",
    "ruột", "lòng nồi",
    "khay", "giỏ", "lưới",
    "đầu", "đầu hút", "đầu xay", 
    "đế", "chân đế",
    "dây nguồn", "phích cắm"
]

accessory_generic = [
    "phụ kiện", "linh kiện", "đồ thay thế",
    "replacement", "spare", "part",
    "combo phụ kiện", "bộ phụ kiện",
    "kit", "set", "bộ",
]

accessory_keywords = list(set(
    accessory_phone +
    accessory_laptop +
    accessory_appliance +
    accessory_home +
    accessory_generic
))

device_keywords = [
    "điện thoại", "iphone", "samsung",
    "laptop", "macbook", "máy tính", "pc",
    "máy giặt", "máy lạnh", "điều hòa",
    "tủ lạnh", "tivi",
    "máy", "bếp", "lò"
]

def count_kw(text, keywords):
    return sum(kw in text for kw in keywords)

### TF-IDF cho từ khóa theo category

Sử dụng `TfidfVectorizer` (sklearn) trên tên/mô tả sản phẩm để:
- Xây dựng ma trận TF-IDF cho toàn bộ sản phẩm
- Tính TF-IDF trung bình của từng từ trong từng `category_label`
- In ra top từ khóa có TF-IDF cao nhất (đặc trưng nhất) cho mỗi category.

In [238]:
# 1. Chuẩn bị data + text
df = df_product[["product_id", "category_id", "product_name", "brand", "description"]].copy()

df["text"] = (
    df["product_name"].fillna("") + " " +
    df["brand"].fillna("") + " " +
    df["description"].fillna("")
)

# clean text
df["text"] = (
    df["text"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)


df = df.dropna(subset=["category_id", "text"])

# 2. Lọc category ít data
valid_cats = df["category_id"].value_counts()
valid_cats = valid_cats[valid_cats > 5].index
df = df[df["category_id"].isin(valid_cats)]

# 3. TF-IDF (có bigram)
vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df["text"])

# 4. Tính centroid mỗi category
cat_ids = df["category_id"].unique()
centroids = {
    cat: np.asarray(X[df["category_id"].values == cat].mean(axis=0)).reshape(1, -1)
    for cat in cat_ids
}

centroid_matrix = np.vstack([centroids[c] for c in cat_ids])


# 5. Similarity matrix
sim_matrix = cosine_similarity(X, centroid_matrix)

# 6. Predict category
best_idx = sim_matrix.argmax(axis=1)
df["category_id_predict"] = [cat_ids[i] for i in best_idx]

# similarity với category hiện tại
cat_id_to_idx = {c: i for i, c in enumerate(cat_ids)}

# convert category_id thành index tương ứng
current_idx = df["category_id"].map(cat_id_to_idx).values

# lấy similarity đúng vị trí
df["similarity"] = sim_matrix[np.arange(len(df)), current_idx]

# 7. Margin (độ chắc chắn)
top2 = np.sort(sim_matrix, axis=1)[:, -2:]
df["margin"] = top2[:, 1] - top2[:, 0]

# 8. Flag (kết hợp nhiều điều kiện)
flagged = df[
    (df["similarity"] < 0.05) &
    (df["margin"] < 0.1) &
    (df["category_id"] != df["category_id_predict"])
]

# 9. Map category name
cat_map = df_category[["category_id", "category_name"]].drop_duplicates()

flagged = flagged.merge(
    cat_map.rename(columns={
        "category_id": "category_id_current",
        "category_name": "category_name_current"
    }),
    left_on="category_id",
    right_on="category_id_current",
    how="left"
)

flagged = flagged.merge(
    cat_map.rename(columns={
        "category_id": "category_id_predict",
        "category_name": "category_name_predict"
    }),
    on="category_id_predict",
    how="left"
)

# 10. Output
# === OUTPUT TOP-K SẢN PHẨM NGHI NGỜ ===

# 1. mismatch
df["is_mismatch"] = df["category_id"] != df["category_id_predict"]

# 2. similarity với category predict
df["sim_pred"] = sim_matrix.max(axis=1)

# 3. detect type


df["accessory_score"] = df["text"].apply(lambda x: count_kw(x, accessory_keywords))
df["device_score"] = df["text"].apply(lambda x: count_kw(x, device_keywords))
df["is_pure_accessory"] = (
    (df["accessory_score"] > 0) &
    (df["device_score"] == 0)
)

# 4. RULE FILTER (loại false positive)
df["is_false_positive"] = (
    df["is_pure_accessory"] & 
    df["is_mismatch"]
)

# 5. score (càng thấp càng đáng nghi)
df["score"] = (
    df["similarity"] 
    - df["sim_pred"] 
    + 0.5 * df["margin"]
    + 0.05 * df["accessory_score"]  
)



# 6. candidate (đã filter)
candidates = df[
    (df["is_mismatch"]) &
    (~df["is_false_positive"]) &  # loại phụ kiện bị kéo sai
    (df["similarity"] < 0.2) &   # giữ quality
    (df["margin"] < 0.15)
].copy()

# 7. lấy top
top_k = 50
candidates = candidates.sort_values("score").head(top_k)

# 8. map category name
cat_map = df_category[["category_id", "category_name"]].drop_duplicates()

result = candidates.merge(
    cat_map.rename(columns={
        "category_id": "category_id_current",
        "category_name": "category_name_current"
    }),
    left_on="category_id",
    right_on="category_id_current",
    how="left"
)

result = result.merge(
    cat_map.rename(columns={
        "category_id": "category_id_predict",
        "category_name": "category_name_predict"
    }),
    on="category_id_predict",
    how="left"
)

# 9. select columns
result = result[[
    "product_id",
    "product_name",
    "category_id_current",
    "category_name_current",
    "category_id_predict",
    "category_name_predict",
    "similarity",
    "margin",
    "score"
]]

# 10. display
#pd.set_option("display.max_rows", 100)

print(f"Số product nghi ngờ (top {top_k}): {len(result)}")
display(result)

Số product nghi ngờ (top 50): 50


,product_id,product_name,category_id_current,category_name_current,category_id_predict,category_name_predict,similarity,margin,score
0,276135770,máy rang hạt cà phê công nghệ gia nhiệt không ...,8728,Đồ dùng nhà bếp khác,24050,Máy xay cà phê,0.159401,0.116175,-0.529377
1,275061940,máy rang hạt cà phê công nghệ gia nhiệt không ...,8728,Đồ dùng nhà bếp khác,24050,Máy xay cà phê,0.174231,0.117744,-0.517457
2,278907834,máy rang hạt cà phê và các loại hạt tự động th...,8728,Đồ dùng nhà bếp khác,24050,Máy xay cà phê,0.154691,0.130388,-0.512842
3,276611166,máy rửa chén kaff kf-bdwsi12.6 - hàng chính hãng,8728,Đồ dùng nhà bếp khác,67223,Máy rửa chén bán âm,0.193503,0.044726,-0.496075
4,278279719,máy rửa chén kaff kf-sbl775b new plus - hàng...,8728,Đồ dùng nhà bếp khác,67223,Máy rửa chén bán âm,0.195415,0.028548,-0.462433
5,48583111,"bảo vệ tủ lạnh, thiết bị lạnh với standa bảo v...",28892,"Phụ kiện, linh kiện điện lạnh khác",67194,Tủ lạnh mini,0.154094,0.039973,-0.461688
6,275906423,máy rửa chén kf-w45a1a401j. hàng chính hãng,23818,Máy sấy chén,67223,Máy rửa chén bán âm,0.192771,0.017329,-0.458526
7,275705476,vòi rửa chén bát inox 206806a (hàng nhập khẩu),8728,Đồ dùng nhà bếp khác,67219,Máy rửa chén độc lập,0.176884,0.039735,-0.447436
8,273168047,giá treo ly euronox en10. hàng chính hãng,5263,Bàn Phím Văn Phòng Có Dây,8070,Giá treo loa,0.054187,0.063488,-0.447221
9,276611207,máy rửa chén kf-w60c3a401l - hàng chính hãng,23818,Máy sấy chén,67223,Máy rửa chén bán âm,0.177633,0.000475,-0.446653


Chúng em đã thử nghiệm qua các mức giá trị thì 50-100 top score đầu sẽ cho ra tỉ lệ chính xác cao nhất.
Khó khăn:
- Do hạn chế của phương pháp lọc cơ bản.
- Sự đa dạng trong tên sản phẩm.
- Có nhiều category tương tự nhau (Mà sản phẩm có thể thuộc về 1 trong n category đó một cách hợp lý)

- Lý do chọn phương pháp này: không cần công cụ phức tạp, thời gian chạy lâu.
- Độ chính xác tốt, tuy không xuất sắc như các phương pháp ứng dụng LLM nhưng vẫn cho ra được kết quả tốt. Có tính ứng dụng cao trong việc phân loại sp vào đúng cateogry

# Sau khi duyệt lại list, đây là list các sản phẩm bị gán nhầm category:

## Các sản phẩm thực sự sai category (lọc từ top 50 nghi ngờ phía trên)

- Cách lọc: kiểm tra thủ công, ưu tiên các trường hợp sai rõ ràng và ít mơ hồ.
- Số lượng xác nhận sai: 18 / 50

## Ghi chú
- Các trường hợp còn lại trong top 50 là mơ hồ hoặc có thể đúng theo cách phân loại hiện tại (không kết luận sai ngay). Ví dụ: "giá treo màn hình lcd brateck ldt69-c012uc (17’’- 49’’) – hàng chính hãng" có thể là "Giá Đỡ - Chân Đế Thường" theo cách phân loại hiện tại của tiki, nhưng theo TF-IDF prediction là "Giá treo Tivi" cũng hợp lý. Do đó, những trường hợp này sẽ được giữ nguyên category.

- Một số sản phẩm có cả category hiện tại và category được đề xuất đều sai. Ví dụ: "giá treo ly euronox en10. hàng chính hãng" có category hiện tại là "Bàn Phím Văn Phòng Có Dây" (rõ ràng sai) và category đề xuất là "Giá treo loa" (cũng không chính xác). Trong trường hợp này, chúng em sẽ chọn tra trên tiki và chọn category phù hợp nhất với sản phẩm (ở trường hợp này là Đồ dùng nhà bếp khác).

---

| stt | product_id | product_name | current_category_id | current_category | suggested_category_id | suggested_category |
|---:|---:|---|---:|---|---:|---|
| 1 | 276611166 | máy rửa chén kaff kf-bdwsi12.6 - hàng chính hãng | 8728 | Đồ dùng nhà bếp khác | 67223 | Máy rửa chén bán âm |
| 2 | 278279719 | máy rửa chén kaff kf-sbl775b new plus  -  hàng chính hãng | 8728 | Đồ dùng nhà bếp khác | 67223 | Máy rửa chén bán âm |
| 3 | 275906423 | máy rửa chén kf-w45a1a401j. hàng chính hãng | 23818 | Máy sấy chén | 67223 | Máy rửa chén bán âm | 
| 4 | 273168047 | giá treo ly euronox en10. hàng chính hãng | 5263 | Bàn Phím Văn Phòng Có Dây | 8728 | Đồ dùng nhà bếp khác |
| 5 | 276611207 | máy rửa chén kf-w60c3a401l - hàng chính hãng | 23818 | Máy sấy chén | 67223 | Máy rửa chén bán âm |
| 6 | 178575599 | dây sạc thay thế cho máy cạo râu enchen, tông đơ enchen, tăm nước enchen - hàng chính hãng | 12294 | Dây Cáp Sạc USB Type-C | 55949 | Phụ kiện máy cạo râu, cắt tóc |
| 7 | 274924699 | máy rửa bát sóng siêu âm utc 1800hd chậu đôi - hàng chính hãng | 8728 | Đồ dùng nhà bếp khác | 67220 | Máy rửa chén để bàn | 
| 8 | 11617909 | micro không dây cho máy ảnh, máy quay comica cvm-wm100 - hàng chính hãng | 20174 | Cáp Âm Thanh | 28644 | Microphone Di Động | 
| 9 | 277356315 | bút vẽ wacom pen 4k lp-1100k cho các dòng bảng vẽ wacom intuos ctl-4100/4100wl/6100wl- hàng chính hãng | 10413 | Đồ Chơi Công Nghệ - Thiết Bị Số và Phụ Kiện Số Khác | 28708 | Bút Vẽ Đồ Họa Cảm Ứng |
| 10 | 11103031 | bộ thu sóng wifi lb-link bl-wn151 - hàng chính hãng | 4642 | Phụ Kiện Thiết Bị Mạng | 4294 | USB Wifi |
| 11 | 198424590 | máy cạo râu du lịch cầm tay philips s1301/02 | 2334 | Máy sấy tóc | 1989 | Máy cạo râu |
| 12 | 192880301 | dây sạc thay thế cho máy cạo râu enchen, tông đơ enchen, tăm nước enchen - hàng chính hãng | 12294 | Dây Cáp Sạc USB Type-C | 55949 | Phụ kiện máy cạo râu, cắt tóc |
| 13 | 279185974 | tai nghe nhét tai samsung eo-ic100 type c - hàng chính hãng | 1795 | Điện thoại Smartphone | 4428 | Tai Nghe Có Dây Nhét Tai |
| 14 | 278856227 | [gift] hộp sấy giày lg pdkacc01.astd - hàng chính hãng | 2010 | Máy lọc không khí | 67209 | Tủ sấy quần áo |
| 15 | 275947327 | tai nghe chụp tai gamming a4 dây 3.5mm- hàng chính hãng | 1819 | Phụ Kiện Di Động Khác | 2977 | Tai Nghe Có Dây Chụp Tai (Over-Ear) |
| 16 | 216116698 | razer leviathan v2 x dàn âm thanh chơi game pc - hàng nhập khẩu | 8066 | Loa Kéo | 8067 | Loa thanh, Soundbar |
| 17 | 270908204 | cáp iphone 1.2m 20w dream - pisen quick dream molding pd ( type-c to light ning ) (dm-tc16) - hàng chính hãng | 4593 | Hub Chuyển Đổi USB Type-C | 5006 | Dây Cáp Sạc iPhone, iPad |
| 18 | 278213719 | máy tỉa lông mũi - hàng nhập khẩu | 1989 | Máy cạo râu | 55947 | Máy tỉa lông mũi |

In [ ]:
#gán đúng category cho các product đã xác nhận sai category

# product_id -> category_id đã xác nhận đúng
confirmed_category_fix = {
    276611166: 67223,
    278279719: 67223,
    275906423: 67223,
    273168047: 8728,
    276611207: 67223,
    178575599: 55949,
    274924699: 67220,
    11617909: 28644,
    277356315: 28708,
    11103031: 4294,
    198424590: 1989,
    192880301: 55949,
    279185974: 4428,
    278856227: 67209,
    275947327: 2977,
    216116698: 8067,
    270908204: 5006,
    278213719: 55947,
}

save_path = Path("../data/products.csv")

fix_ids = set(confirmed_category_fix.keys())
mask = df_product["product_id"].isin(fix_ids)

found_ids = set(df_product.loc[mask, "product_id"].unique().tolist())
missing_ids = sorted(fix_ids - found_ids)
if missing_ids:
    print("không tìm thấy một số product_id trong df_product:", missing_ids)

before_df = df_product.loc[mask, ["product_id", "product_name", "category_id"]].copy()
before_df = before_df.rename(columns={"category_id": "old_category_id"})

df_product.loc[mask, "category_id"] = df_product.loc[mask, "product_id"].map(confirmed_category_fix).astype(int)

after_df = df_product.loc[mask, ["product_id", "category_id"]].copy()
after_df = after_df.rename(columns={"category_id": "new_category_id"})

audit_df = before_df.merge(after_df, on="product_id", how="left")

cat_lookup = df_category[["category_id", "category_name"]].drop_duplicates()
audit_df = audit_df.merge(
    cat_lookup.rename(columns={"category_id": "old_category_id", "category_name": "old_category_name"}),
    on="old_category_id",
    how="left",
)
audit_df = audit_df.merge(
    cat_lookup.rename(columns={"category_id": "new_category_id", "category_name": "new_category_name"}),
    on="new_category_id",
    how="left",
)

audit_df = audit_df[[
    "product_id",
    "product_name",
    "old_category_id",
    "old_category_name",
    "new_category_id",
    "new_category_name",
]]

print(f"Số product_id cần sửa: {len(fix_ids)}")
print(f"Số product_id tìm thấy trong df_product: {len(found_ids)}")
print(f"Số dòng được cập nhật trong df_product: {int(mask.sum())}")

display(audit_df.sort_values("product_id").reset_index(drop=True))

# Lưu lại vào products.csv
df_product.to_csv(save_path, index=False)
print(f"Đã lưu products đã cập nhật vào: {save_path}")

Số product_id cần sửa: 18
Số product_id tìm thấy trong df_product: 18
Số dòng được cập nhật trong df_product: 18


,product_id,product_name,old_category_id,old_category_name,new_category_id,new_category_name
0,11103031,bộ thu sóng wifi lb-link bl-wn151 - hàng chính...,4642,Phụ Kiện Thiết Bị Mạng,4294,USB Wifi
1,11617909,"micro không dây cho máy ảnh, máy quay comica c...",20174,Cáp Âm Thanh,28644,Microphone Di Động
2,178575599,"dây sạc thay thế cho máy cạo râu enchen, tông ...",12294,Dây Cáp Sạc USB Type-C,55949,"Phụ kiện máy cạo râu, cắt tóc"
3,192880301,"dây sạc thay thế cho máy cạo râu enchen, tông ...",12294,Dây Cáp Sạc USB Type-C,55949,"Phụ kiện máy cạo râu, cắt tóc"
4,198424590,máy cạo râu du lịch cầm tay philips s1301/02,2334,Máy sấy tóc,1989,Máy cạo râu
5,216116698,razer leviathan v2 x dàn âm thanh chơi game pc...,8066,Loa Kéo,8067,"Loa thanh, Soundbar"
6,270908204,cáp iphone 1.2m 20w dream - pisen quick dream ...,4593,Hub Chuyển Đổi USB Type-C,5006,"Dây Cáp Sạc iPhone, iPad"
7,273168047,giá treo ly euronox en10. hàng chính hãng,5263,Bàn Phím Văn Phòng Có Dây,8728,Đồ dùng nhà bếp khác
8,274924699,máy rửa bát sóng siêu âm utc 1800hd chậu đôi -...,8728,Đồ dùng nhà bếp khác,67220,Máy rửa chén để bàn
9,275906423,máy rửa chén kf-w45a1a401j. hàng chính hãng,23818,Máy sấy chén,67223,Máy rửa chén bán âm


Đã lưu products đã cập nhật vào: ../data/products.csv
